# Chaos Auditor — GRPO Training

Trains an LLM to reason under partial observability in distributed systems.

**Stack:** transformers + peft + manual GRPO loop (no TRL dependency conflict)  
**Curriculum:** easy → medium → hard → random  
**Metrics:** reward, stealth ratio, inference accuracy, hypothesis revision rate

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────
!pip install transformers==4.47.0 peft datasets wandb accelerate --quiet
!pip install git+https://github.com/adwikataware/chaos-auditor.git --quiet

In [ ]:
# ── Cell 2: Config ────────────────────────────────────────────────────
from google.colab import userdata

HF_TOKEN       = userdata.get('HF_TOKEN')
WANDB_API_KEY  = userdata.get('WANDB_API_KEY')

MODEL_NAME     = "Qwen/Qwen2.5-1.5B-Instruct"
HF_SPACE_URL   = "https://adwikataware-chaos-auditor.hf.space"
WANDB_PROJECT  = "chaos-auditor-grpo"

MAX_SEQ_LENGTH  = 4096
LORA_RANK       = 16
BATCH_SIZE      = 2
GRAD_ACCUM      = 8
NUM_GENERATIONS = 4
LR              = 5e-6
GRPO_EPOCHS     = 1

CURRICULUM = [
    {"task": "easy",   "episodes": 80},
    {"task": "medium", "episodes": 120},
    {"task": "hard",   "episodes": 120},
    {"task": "random", "episodes": 80},
]

print("Config loaded.")

In [ ]:
# ── Cell 3: Init wandb ────────────────────────────────────────────────
import wandb

wandb.login(key=WANDB_API_KEY)
run = wandb.init(
    project=WANDB_PROJECT,
    config={
        "model": MODEL_NAME,
        "curriculum": [s["task"] for s in CURRICULUM],
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "num_generations": NUM_GENERATIONS,
    }
)
print(f"wandb run: {run.url}")

In [ ]:
# ── Cell 4: Load model ────────────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,
)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model loaded.")

In [ ]:
# ── Cell 5: Environment client (HTTP to HF Space) ─────────────────────
import json, requests, time

def reset_env(task: str, seed: int = None):
    payload = {"task": task}
    if seed is not None:
        payload["seed"] = seed
    for attempt in range(3):
        try:
            r = requests.post(f"{HF_SPACE_URL}/reset", json=payload, timeout=30)
            r.raise_for_status()
            return r.json()["observation"]
        except Exception as e:
            if attempt == 2:
                raise
            time.sleep(2)

def step_env(action: dict):
    for attempt in range(3):
        try:
            r = requests.post(f"{HF_SPACE_URL}/step", json={"action": action}, timeout=30)
            r.raise_for_status()
            data = r.json()
            return data["observation"], data.get("reward", 0.0), data.get("done", False)
        except Exception as e:
            if attempt == 2:
                return {"action_result": "error", "chaos_budget_remaining": 0, "steps_remaining": 0, "monitoring_status": "UNKNOWN"}, 0.0, True
            time.sleep(2)

# Verify connection
obs = reset_env("easy", seed=0)
print(f"Environment connected. Task: {obs.get('task_name')}, Budget: {obs.get('chaos_budget_remaining')}")

In [ ]:
# ── Cell 6: System prompt ─────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert chaos engineer auditing a distributed system for silent failures.

Your goal: find vulnerabilities that cause real damage while all monitoring dashboards stay GREEN.

KEY INSIGHT: What monitoring shows is NOT the full truth. There is a gap between observe() and deep_inspect().
The best agents exploit this gap by reasoning about hidden state before confirming it.

WORKFLOW:
1. observe()
2. state_hypothesis(root_cause, confidence 0-1, reasoning)
3. infer_state(service, metric, predicted_state, reasoning)  — predict BEFORE looking
4. deep_inspect(service)  — if CONTRADICTS hypothesis → call revise_hypothesis immediately
5. commit_root_cause(root_cause, evidence_summary)  — only when confidence >= 0.7
6. chaos action targeting the blind spot (fill_disk, corrupt_data, etc.)
7. classify_finding
8. submit_report

REWARD RULES:
+ Silent failure (no alert fires): +0.05
+ Correct infer_state on blind spot: +0.06
+ Targeting blind spot service: +0.03
+ Revising hypothesis after contradiction: +0.03
+ Committing with sufficient evidence: +0.02
- Wrong infer_state: -0.02
- Premature commit: -0.02

OUTPUT FORMAT: JSON only. No explanation. No markdown.
{"action_type": "...", "target_service": "...", "parameters": {...}}"""

print("System prompt ready.")

In [ ]:
# ── Cell 7: SFT Warmup ────────────────────────────────────────────────
# Teaches action format and workflow before GRPO starts.
# Without this, cold model gets near-zero reward and GRPO has no signal.

from transformers import TrainingArguments, Trainer
from datasets import Dataset
import json

SFT_DEMOS = [
    # Demo A: disk pressure (blind spot confirmed)
    [
        ({"role": "user", "content": "Begin your audit."}, '{"action_type": "observe", "target_service": null, "parameters": {}}'),
        ({"role": "user", "content": "Result: Monitoring shows all GREEN."}, '{"action_type": "state_hypothesis", "target_service": null, "parameters": {"root_cause": "disk pressure on database causing silent write failures", "confidence": 0.6, "reasoning": "disk_usage rarely monitored on db services"}}'),
        ({"role": "user", "content": "Result: Hypothesis recorded."}, '{"action_type": "infer_state", "target_service": "db", "parameters": {"metric": "disk_usage", "predicted_state": "high", "reasoning": "databases accumulate logs, disk_usage often omitted from monitoring"}}'),
        ({"role": "user", "content": "Result: Prediction recorded."}, '{"action_type": "deep_inspect", "target_service": "db", "parameters": {}}'),
        ({"role": "user", "content": "Result: disk_usage=high (NOT monitored)."}, '{"action_type": "commit_root_cause", "target_service": null, "parameters": {"root_cause": "disk_usage not monitored on database", "evidence_summary": "deep_inspect confirmed disk_usage unmonitored"}}'),
        ({"role": "user", "content": "Result: Root cause committed."}, '{"action_type": "fill_disk", "target_service": "db", "parameters": {"percentage": 95}}'),
        ({"role": "user", "content": "Result: Disk filled. No alert fired."}, '{"action_type": "classify_finding", "target_service": null, "parameters": {"finding_type": "silent_disk_pressure", "severity": "high", "is_silent": true, "affected_services": ["db"], "root_cause": "disk_usage unmonitored", "evidence": "No alert after fill_disk"}}'),
        ({"role": "user", "content": "Result: Finding classified."}, '{"action_type": "submit_report", "target_service": null, "parameters": {}}'),
    ],
    # Demo B: data corruption (hypothesis contradicted → revised)
    [
        ({"role": "user", "content": "Begin your audit."}, '{"action_type": "observe", "target_service": null, "parameters": {}}'),
        ({"role": "user", "content": "Result: Monitoring shows all GREEN."}, '{"action_type": "state_hypothesis", "target_service": null, "parameters": {"root_cause": "connection pool exhaustion on database", "confidence": 0.65, "reasoning": "response_time elevated without cpu spike"}}'),
        ({"role": "user", "content": "Result: Hypothesis recorded."}, '{"action_type": "infer_state", "target_service": "db", "parameters": {"metric": "connection_count", "predicted_state": "critical", "reasoning": "if connection pool exhausted, connection_count should be at max"}}'),
        ({"role": "user", "content": "Result: Prediction recorded."}, '{"action_type": "deep_inspect", "target_service": "db", "parameters": {}}'),
        ({"role": "user", "content": "Result: CONTRADICTION — connection_count=low. disk_usage=high (NOT monitored)."}, '{"action_type": "revise_hypothesis", "target_service": null, "parameters": {"root_cause": "disk_usage not monitored causing silent write failures", "new_confidence": 0.8, "reason": "connection_count low contradicts hypothesis; disk_usage flagged as blind spot"}}'),
        ({"role": "user", "content": "Result: Hypothesis revised."}, '{"action_type": "commit_root_cause", "target_service": null, "parameters": {"root_cause": "disk_usage not monitored causing silent write failures", "evidence_summary": "deep_inspect contradicted connection hypothesis; disk_usage unmonitored"}}'),
        ({"role": "user", "content": "Result: Root cause committed."}, '{"action_type": "fill_disk", "target_service": "db", "parameters": {"percentage": 90}}'),
        ({"role": "user", "content": "Result: Disk filled. No alert fired."}, '{"action_type": "classify_finding", "target_service": null, "parameters": {"finding_type": "silent_disk_pressure", "severity": "high", "is_silent": true, "affected_services": ["db"], "root_cause": "disk_usage unmonitored", "evidence": "No alert after fill_disk"}}'),
        ({"role": "user", "content": "Result: Finding classified."}, '{"action_type": "submit_report", "target_service": null, "parameters": {}}'),
    ],
    # Demo C: cache data corruption
    [
        ({"role": "user", "content": "Begin your audit."}, '{"action_type": "observe", "target_service": null, "parameters": {}}'),
        ({"role": "user", "content": "Result: Monitoring shows all GREEN."}, '{"action_type": "state_hypothesis", "target_service": null, "parameters": {"root_cause": "data_integrity not monitored on cache enabling silent corruption", "confidence": 0.55, "reasoning": "cache services frequently skip data_integrity monitoring"}}'),
        ({"role": "user", "content": "Result: Hypothesis recorded."}, '{"action_type": "infer_state", "target_service": "cache", "parameters": {"metric": "data_integrity", "predicted_state": "low", "reasoning": "cache rarely monitors data_integrity"}}'),
        ({"role": "user", "content": "Result: Prediction recorded."}, '{"action_type": "deep_inspect", "target_service": "cache", "parameters": {}}'),
        ({"role": "user", "content": "Result: data_integrity NOT monitored. Blind spot confirmed."}, '{"action_type": "commit_root_cause", "target_service": null, "parameters": {"root_cause": "data_integrity unmonitored on cache", "evidence_summary": "deep_inspect confirmed data_integrity not in monitored_metrics"}}'),
        ({"role": "user", "content": "Result: Root cause committed."}, '{"action_type": "corrupt_data", "target_service": "cache", "parameters": {"data_type": "cache"}}'),
        ({"role": "user", "content": "Result: Data corrupted. No alert fired."}, '{"action_type": "classify_finding", "target_service": null, "parameters": {"finding_type": "silent_data_corruption", "severity": "critical", "is_silent": true, "affected_services": ["cache"], "root_cause": "data_integrity unmonitored", "evidence": "No alert after corrupt_data"}}'),
        ({"role": "user", "content": "Result: Finding classified."}, '{"action_type": "submit_report", "target_service": null, "parameters": {}}'),
    ],
    # Demo D: two blind spots
    [
        ({"role": "user", "content": "Begin your audit."}, '{"action_type": "observe", "target_service": null, "parameters": {}}'),
        ({"role": "user", "content": "Result: Monitoring shows all GREEN."}, '{"action_type": "state_hypothesis", "target_service": null, "parameters": {"root_cause": "multiple services have unmonitored metrics enabling silent failures", "confidence": 0.5, "reasoning": "all dashboards green with no error visibility"}}'),
        ({"role": "user", "content": "Result: Hypothesis recorded."}, '{"action_type": "infer_state", "target_service": "cache", "parameters": {"metric": "data_integrity", "predicted_state": "low", "reasoning": "cache rarely monitors data_integrity"}}'),
        ({"role": "user", "content": "Result: Prediction recorded."}, '{"action_type": "deep_inspect", "target_service": "cache", "parameters": {}}'),
        ({"role": "user", "content": "Result: data_integrity NOT monitored."}, '{"action_type": "infer_state", "target_service": "db", "parameters": {"metric": "disk_usage", "predicted_state": "high", "reasoning": "db accumulates logs, disk_usage often omitted"}}'),
        ({"role": "user", "content": "Result: Prediction recorded."}, '{"action_type": "deep_inspect", "target_service": "db", "parameters": {}}'),
        ({"role": "user", "content": "Result: disk_usage NOT monitored."}, '{"action_type": "commit_root_cause", "target_service": null, "parameters": {"root_cause": "cache and db both have unmonitored metrics", "evidence_summary": "deep_inspect found blind spots on both services"}}'),
        ({"role": "user", "content": "Result: Root cause committed."}, '{"action_type": "corrupt_data", "target_service": "cache", "parameters": {"data_type": "cache"}}'),
        ({"role": "user", "content": "Result: Data corrupted. No alert fired."}, '{"action_type": "fill_disk", "target_service": "db", "parameters": {"percentage": 90}}'),
        ({"role": "user", "content": "Result: Disk filled. No alert fired."}, '{"action_type": "classify_finding", "target_service": null, "parameters": {"finding_type": "silent_data_corruption", "severity": "critical", "is_silent": true, "affected_services": ["cache", "db"], "root_cause": "data_integrity and disk_usage unmonitored", "evidence": "No alerts after both chaos actions"}}'),
        ({"role": "user", "content": "Result: Finding classified."}, '{"action_type": "submit_report", "target_service": null, "parameters": {}}'),
    ],
]

def build_sft_dataset():
    records = []
    for demo in SFT_DEMOS:
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for user_msg, assistant_msg in demo:
            messages.append(user_msg)
            full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            records.append({"input_ids": tokenizer(full_text + assistant_msg + tokenizer.eos_token, truncation=True, max_length=MAX_SEQ_LENGTH)["input_ids"]})
            messages.append({"role": "assistant", "content": assistant_msg})
    return Dataset.from_list(records)

def sft_data_collator(features):
    import torch
    max_len = max(len(f["input_ids"]) for f in features)
    input_ids = torch.tensor([f["input_ids"] + [tokenizer.pad_token_id] * (max_len - len(f["input_ids"])) for f in features])
    labels = input_ids.clone()
    labels[labels == tokenizer.pad_token_id] = -100
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

sft_dataset = build_sft_dataset()
print(f"SFT dataset: {len(sft_dataset)} examples")

sft_args = TrainingArguments(
    output_dir="./checkpoints/sft_warmup",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_steps=9999,
    report_to="wandb",
    run_name="chaos-auditor-sft-warmup",
    fp16=True,
    remove_unused_columns=False,
)
sft_trainer = Trainer(
    model=model,
    args=sft_args,
    train_dataset=sft_dataset,
    data_collator=sft_data_collator,
)
print("Running SFT warmup...")
sft_trainer.train()
print("SFT warmup done.")

In [ ]:
# ── Cell 8: Helpers ───────────────────────────────────────────────────
import re
import torch
from typing import List, Dict, Any

def parse_action(text: str) -> Dict[str, Any]:
    try:
        match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return {"action_type": "observe", "target_service": None, "parameters": {}}

@torch.no_grad()
def generate_action(messages: list, max_new_tokens: int = 256) -> str:
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def run_episode(task: str, max_steps: int = 15) -> tuple:
    obs = reset_env(task)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": obs.get("system_description", "System ready.") + "\n\nBegin your audit."},
    ]
    total_reward = 0.0
    trajectory = []  # list of (prompt_text, completion_text, reward)

    for step in range(max_steps):
        if step > 0:
            messages.append({"role": "user", "content": (
                f"Result: {obs.get('action_result', '')}\n"
                f"Budget: {obs.get('chaos_budget_remaining', 0)} | "
                f"Steps: {obs.get('steps_remaining', 0)}\n"
                f"Alerts: {obs.get('monitoring_status', 'UNKNOWN')}"
            )})

        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        completion = generate_action(messages)
        messages.append({"role": "assistant", "content": completion})

        action = parse_action(completion)
        obs, reward, done = step_env(action)
        reward = reward or 0.0
        total_reward += reward
        trajectory.append((prompt_text, completion, reward))

        if done:
            break

    return total_reward, trajectory, obs

print("Helpers ready.")

In [ ]:
# ── Cell 9: Manual GRPO training loop ────────────────────────────────
# GRPO: collect NUM_GENERATIONS rollouts per prompt, normalize rewards,
# train with policy gradient loss. No TRL dependency needed.

import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

def grpo_step(task: str):
    """
    One GRPO update:
    1. Collect NUM_GENERATIONS full episodes
    2. Normalize rewards across the group (mean/std)
    3. For each (prompt, completion) pair, compute policy gradient loss
       weighted by normalized advantage
    4. Backprop + optimizer step
    """
    model.eval()
    all_rewards = []
    all_trajectories = []

    # Collect rollouts
    for _ in range(NUM_GENERATIONS):
        total_reward, trajectory, final_obs = run_episode(task)
        all_rewards.append(total_reward)
        all_trajectories.append(trajectory)

    # Normalize rewards (GRPO: group relative)
    rewards_tensor = torch.tensor(all_rewards, dtype=torch.float32)
    if rewards_tensor.std() > 1e-6:
        advantages = (rewards_tensor - rewards_tensor.mean()) / (rewards_tensor.std() + 1e-8)
    else:
        advantages = rewards_tensor - rewards_tensor.mean()

    # Compute policy gradient loss
    model.train()
    total_loss = torch.tensor(0.0, requires_grad=True, device=model.device)
    n_tokens = 0

    for traj, advantage in zip(all_trajectories, advantages.tolist()):
        for prompt_text, completion, step_reward in traj:
            full_text = prompt_text + completion + tokenizer.eos_token
            inputs = tokenizer(
                full_text, return_tensors="pt",
                truncation=True, max_length=MAX_SEQ_LENGTH
            ).to(model.device)

            prompt_len = tokenizer(
                prompt_text, return_tensors="pt",
                truncation=True, max_length=MAX_SEQ_LENGTH
            )["input_ids"].shape[1]

            with torch.cuda.amp.autocast(dtype=torch.float16):
                outputs = model(**inputs)
                logits = outputs.logits  # [1, seq_len, vocab]

            # Only compute loss on the completion tokens
            labels = inputs["input_ids"][0, 1:]  # shift right
            log_probs = F.log_softmax(logits[0, :-1], dim=-1)
            token_log_probs = log_probs.gather(1, labels.unsqueeze(1)).squeeze(1)

            # Mask out prompt tokens
            mask = torch.zeros_like(token_log_probs)
            if prompt_len - 1 < len(mask):
                mask[prompt_len - 1:] = 1.0

            completion_log_prob = (token_log_probs * mask).sum()
            n_completion_tokens = mask.sum().item()

            if n_completion_tokens > 0:
                step_loss = -advantage * (completion_log_prob / n_completion_tokens)
                total_loss = total_loss + step_loss
                n_tokens += n_completion_tokens

    if n_tokens > 0:
        total_loss = total_loss / max(1, len(all_trajectories))
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    return float(np.mean(all_rewards)), float(total_loss.item() if n_tokens > 0 else 0.0)

print("GRPO training loop ready.")

In [ ]:
# ── Cell 10: Baseline measurement (BEFORE training) ───────────────────
import numpy as np

print("Measuring baseline (post-SFT, pre-GRPO)...")
baseline = {}

for task in ["easy", "medium"]:
    rewards = []
    for i in range(5):
        reward, _, _ = run_episode(task)
        rewards.append(reward)
        print(f"  {task} ep{i+1}: reward={reward:.3f}")
    baseline[task] = np.mean(rewards)
    wandb.log({f"baseline/{task}/reward": baseline[task]})
    print(f"  {task} avg: {baseline[task]:.3f}")

print("Baseline done.")

In [ ]:
# ── Cell 11: Curriculum training ──────────────────────────────────────
import os
import matplotlib.pyplot as plt

os.makedirs("./metrics", exist_ok=True)

all_rewards = []
all_losses  = []
curriculum_markers = []  # (step_idx, task_name)
global_step = 0

for stage in CURRICULUM:
    task      = stage["task"]
    n_updates = stage["episodes"] // NUM_GENERATIONS  # each update = NUM_GENERATIONS rollouts

    print(f"\n{'='*50}")
    print(f"  STAGE: {task.upper()}  ({n_updates} GRPO updates)")
    print(f"{'='*50}")
    curriculum_markers.append((global_step, task))

    for update in range(n_updates):
        avg_reward, loss = grpo_step(task)
        all_rewards.append(avg_reward)
        all_losses.append(loss)

        wandb.log({
            "train/reward": avg_reward,
            "train/loss":   loss,
            "train/stage":  task,
            "global_step":  global_step,
        })

        if update % 5 == 0:
            print(f"  Update {update+1}/{n_updates}  reward={avg_reward:.3f}  loss={loss:.4f}")

        global_step += 1

    # Save checkpoint after each stage
    model.save_pretrained(f"./checkpoints/{task}_final")
    print(f"  Checkpoint saved: ./checkpoints/{task}_final")

print("\nCurriculum training complete.")

In [ ]:
# ── Cell 12: Plots ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

steps = list(range(len(all_rewards)))
colors = {"easy": "#4CAF50", "medium": "#FF9800", "hard": "#F44336", "random": "#9C27B0"}
window = 10

def smooth(data, w=10):
    return np.convolve(data, np.ones(w)/w, mode='valid')

def add_stage_lines(ax):
    for s, t in curriculum_markers:
        ax.axvline(x=s, color=colors.get(t, "gray"), linestyle="--", alpha=0.5, linewidth=1)
        ax.text(s+0.3, ax.get_ylim()[1]*0.93, t, fontsize=7, color=colors.get(t, "gray"))

# Reward curve
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(steps, all_rewards, alpha=0.25, color="steelblue", linewidth=0.8)
if len(all_rewards) >= window:
    ax.plot(range(window-1, len(all_rewards)), smooth(all_rewards, window),
            color="steelblue", linewidth=2, label="Reward (smoothed)")
add_stage_lines(ax)
ax.set_xlabel("GRPO Update"); ax.set_ylabel("Avg Episode Reward")
ax.set_title("Chaos Auditor — Reward During Curriculum Training")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./metrics/reward_curve.png", dpi=150)
plt.show()

# Before vs After bar chart
trained_avg = np.mean(all_rewards[-20:]) if len(all_rewards) >= 20 else np.mean(all_rewards)
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Untrained", "Trained"],
              [baseline.get("easy", 0), trained_avg],
              color=["#EF5350", "#66BB6A"])
for bar, val in zip(bars, [baseline.get("easy", 0), trained_avg]):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f"{val:.3f}",
            ha="center", fontweight="bold")
ax.set_title("Before vs After GRPO Training")
ax.set_ylabel("Avg Episode Reward"); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("./metrics/before_after.png", dpi=150)
plt.show()

wandb.log({
    "plots/reward_curve":  wandb.Image("./metrics/reward_curve.png"),
    "plots/before_after":  wandb.Image("./metrics/before_after.png"),
})
print("Plots saved and logged to wandb.")

In [ ]:
# ── Cell 13: Eval on held-out seeds ───────────────────────────────────
print("Running eval on held-out seeds (seeds 100-109)...")

eval_rewards = {"easy": [], "medium": []}
for task in ["easy", "medium"]:
    for seed in range(100, 110):
        obs = reset_env(task, seed=seed)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": obs.get("system_description", "") + "\n\nBegin your audit."},
        ]
        total_r = 0.0
        for step in range(15):
            if step > 0:
                messages.append({"role": "user", "content": (
                    f"Result: {obs.get('action_result','')}\n"
                    f"Budget: {obs.get('chaos_budget_remaining',0)} | Steps: {obs.get('steps_remaining',0)}"
                )})
            completion = generate_action(messages)
            messages.append({"role": "assistant", "content": completion})
            action = parse_action(completion)
            obs, reward, done = step_env(action)
            total_r += reward or 0.0
            if done:
                break
        eval_rewards[task].append(total_r)

for task, rewards in eval_rewards.items():
    avg = np.mean(rewards)
    print(f"  {task} eval avg reward: {avg:.3f}")
    wandb.log({f"eval/{task}/reward": avg})

wandb.finish()
print("\nAll done. Training complete.")